# LLM AS A JUDGE 

In [1]:
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q      
!pip install torch sacrebleu "unbabel-comet>=2.2.0" bert-score evaluate tensorflow numpy nltk -q
!pip install git+https://github.com/google-research/bleurt.git -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch, datetime, re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import sacrebleu                                 
from sacrebleu.metrics import TER                 
from nltk.translate.meteor_score import single_meteor_score  
import numpy as np          
from typing import Union, Dict, List
import warnings

In [3]:
print("Loading Prometheus (4‑bit) …")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "Unbabel/M-Prometheus-7B",
    trust_remote_code=True,
)

model.eval()

Loading Prometheus (4‑bit) …


config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

2025-06-23 00:43:53.272391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750639433.511584      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750639433.578163      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qw

In [4]:
templates = {
        "general":'''###Task Description:
An instruction, a response to evaluate, a **gold translation** (reference answer) and a score rubric representing an evaluation criterion are given.

1. Write a detailed feedback of maximum 100 words that assesses the response strictly according to the score rubric (translation quality), **not evaluating in general**.
2. After writing the feedback, write a score that is an integer between 1 and 5. You must refer to the rubric.
3. The output format must be EXACTLY:
"<your feedback here>
[RESULT] <integer 1-5>"
4. Do not generate any headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Translation Quality):
5 – Excellent: The translation is fluid, precise, and natural. It preserves all the semantic nuances of the original text. The syntax is modern yet respectful of the source content. The text is perfectly readable and does not feel machine‑generated. Archaic terms are rendered with accurate modern equivalents, even in ambiguous cases. Indistinguishable from a translation by a highly qualified human.
4 – Good: A generally correct and readable translation. The text appears modern and understandable, although some expressions come across as slightly mechanical or overly literal. It preserves the overall meaning well but may lose subtle cultural or historical nuances. Some terms are adapted with a good but not optimal choice. It may resemble a solid human translation, but not an outstanding one.
3 – Sufficient: An understandable translation but with evident mechanical traits or stylistically poor passages. There may be rigid constructions, syntactic calques, or unnatural lexical choices. The global meaning is retained, but the expressive richness of the original medieval text is lost. It reads like an automatic translation with minimal human supervision.
2 – Insufficient: A discontinuous or incoherent translation. Sentences are disconnected, modern vocabulary is misused, and ambiguous or obsolete words are misinterpreted. Some passages are incomprehensible or misleading. It clearly feels like a text translated automatically without human intervention. Not reliable for reading or academic study.
1 – Gravely Insufficient: A failed translation. Sentences are nonsensical, terms are mistranslated or omitted entirely. There is no grammatical or semantic coherence. The text is unusable and devoid of communicative value. Neither the original content nor its purpose can be discerned. It seems produced by a crude or faulty automatic system, utterly lacking linguistic or historical understanding.

###Feedback:''',
        "grammatic":'''###Task Description:
An instruction, a model response, a gold translation (reference answer) and a score rubric for grammatical accuracy are provided.

1. Write detailed feedback of no more than 100 words that evaluates the response only against the grammatical rubric (morphology, syntax, agreement, punctuation, orthography). Do not comment on meaning or style.
2. After the feedback, give a score—an integer from 1 to 5—clearly tied to the rubric.
3. The output format must be EXACTLY:
"<your feedback here>
[RESULT] <integer 1-5>"
4. Do not generate headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Grammatic Quality):
5 – Flawless: Grammar, punctuation, and orthography are impeccable. Sentences flow naturally, with varied structure and perfect agreement of tense, number, and gender. Word order feels idiomatic and clear.
4 – Very Good: Only rare, minor slips (e.g., a missing comma or slight tense inconsistency). Fluency and correctness remain high; errors do not interrupt reading.
3 – Adequate: Noticeable but non-disruptive issues such as awkward word order, repeated structures, or several punctuation glitches. Readable but needs copy-editing.
2 – Poor: Frequent grammatical errors (agreement mistakes, run-ons, fragments) distract and sometimes blur meaning. Substantial revision required.
1 – Unacceptable: Pervasive, severe faults make the text hard to follow. Basic syntactic rules are ignored; comprehension is unreliable.

###Feedback:''',
        "semantic":'''###Task Description:
An instruction, a model response, a gold translation (reference answer) and a score rubric for semantic accuracy are provided.

1.Write detailed feedback of no more than 100 words that evaluates the response only against the semantic rubric (fidelity of meaning, preservation of nuance, contextual appropriateness). Do not comment on grammar or style.
2.After the feedback, give a score—an integer from 1 to 5—clearly tied to the rubric.
3.The output format must be EXACTLY:
"<your feedback here>
[RESULT] <integer 1-5>"
4.Do not generate headers, closings, or extra commentary.

###The instruction to evaluate:
Traduci il seguente brano di italiano arcaico nel moderno italiano:
«{ARCHAIC_PASSAGE}»

###Response to evaluate (model output):
{MODEL_TRANSLATION}

###Gold Translation (reference answer):
{GOLD_TRANSLATION}

###Score Rubric (Semantic Quality):
5 – Excellent: All intended meanings, nuances, and relationships are conveyed with precision. Concepts are accurately represented, contextually appropriate, and internally consistent. No misinterpretations or omissions; the text mirrors the source’s full informational depth.
4 – Good: The core meaning and almost all subtleties are preserved. A few minor shades of sense or cultural references may be softened, but factual content and logical relations remain intact. Overall message fidelity is high.
3 – Sufficient: Central ideas are captured, yet several nuances, implications, or culturally loaded elements are diluted or missed. Some statements may feel slightly out of context, but the general message survives without critical distortion.
2 – Insufficient: Major semantic inaccuracies, omissions, or additions alter the author’s intent. Key concepts are confused or simplified, leading to a partial or skewed understanding of the original content.
1 – Gravely Insufficient: Meaning is largely lost or contradicted. Essential ideas are misrepresented, context is ignored, and the text may introduce content not present in the source. Readers cannot rely on the translation to grasp the original message.

###Feedback:'''}
def judge(to_predict, prediction, gold, mode, verbose=0):
    if mode not in set(templates.keys()): raise ValueError("unaccepeted mode")
    
    prompt = templates[mode].format(
        ARCHAIC_PASSAGE=to_predict.strip(),
        MODEL_TRANSLATION=prediction.strip(),
        GOLD_TRANSLATION=gold.strip()
    )

    # 2️⃣  Run Prometheus 2 --------------------------
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    lunghezza_input = len(inputs[0])
    if verbose>0:
        print(f"ORIGINAL:\n {to_predict}")
        print(f"PREDICTION:\n{prediction} ")
        print(f"GOLD:\n {gold}")
        print("▶ Generating judgement ...")
        t0 = datetime.datetime.now()

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens= 256, do_sample=False)

    raw = tokenizer.decode(out[0][lunghezza_input:], skip_special_tokens=True)
    

    if verbose>1:
        print("RAW OUTPUT OF PROMETHEUS:")
        print(raw)
        print("END RAW")

    # 3️⃣  Parse feedback + numeric score ------------
    m = re.search(
        r"^(.*?)\s*\[RESULT\]\s*([1-5])\b",   # ① feedback   ② score
        raw,
        re.S                                   # dot matches newlines
    )
    if not m:
        print("!!!!!!  ERROR: Cannot parse Prometheus output:\n" + raw)
        print(f"Skipping the decomposition of judjment of the sentence: {prediction}")
        feedback = raw
        score    = raw
    else:
        feedback = m.group(1).strip()
        score    = int(m.group(2))
        if verbose>0:
            print(f"SCORE: {score}")
            print("")
        if verbose>1:
            print(f"FEEDBACK: {feedback}")
            print("⏱", datetime.datetime.now() - t0)

    return {"feedback": feedback, "score": score, "raw": raw}

In [5]:
# This is a small trick to avoid reloading models on repeated calls in the same script.
_MODEL_CACHE = {}

def evaluate_metrics(
    original: Union[str, List[str]],
    gold: Union[str, List[str]],
    predicted: Union[str, List[str]]
) -> Union[Dict[str, float], Dict[str, Dict]]:
    """
    A simple, all-in-one function to evaluate MT metrics.

    Handles both single sentences (str) and corpora (List[str]).
    It is resilient to failures in individual metric calculations.

    - For a single sentence, returns a flat dictionary of scores.
    - For a corpus, returns a dictionary where sentence-level metrics
      (BERTScore, BLEURT, COMET-22) contain a 'scores' list with a
      score for each sentence.
    """
    is_corpus = isinstance(gold, list)

    # --- 1. Standardize inputs to lists for uniform processing ---
    if not is_corpus:
        originals, golds, predicteds = [original], [gold], [predicted]
    else:
        if not (isinstance(original, list) and isinstance(predicted, list)):
            raise ValueError("If one input is a list (corpus), all must be lists.")
        originals, golds, predicteds = original, gold, predicted
    
    results = {}
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Running evaluation on {len(golds)} sentence(s) using device: {device}")

    # --- 2. Sacrebleu-based Metrics (fast and reliable) ---
    results['BLEU'] = sacrebleu.corpus_bleu(predicteds, [golds]).score / 100
    results['chrF++'] = sacrebleu.corpus_chrf(predicteds, [golds], word_order=2).score / 100
    ter_score = sacrebleu.corpus_ter(predicteds, [golds]).score / 100
    results['TER (inv)'] = 1.0 - ter_score

    # --- 3. Model-based Metrics (slower, wrapped for safety) ---
    
    # BERTScore
    try:
        if 'bert_scorer' not in _MODEL_CACHE:
            print("Loading BERTScore model...")
            from bert_score import BERTScorer
            _MODEL_CACHE['bert_scorer'] = BERTScorer(lang="it", model_type='bert-base-multilingual-cased', device=device)
        _, _, F1 = _MODEL_CACHE['bert_scorer'].score(predicteds, golds, batch_size=2)
        results['BERTScore'] = F1.tolist() # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate BERTScore: {e}")
        results['BERTScore'] = [-1.0] * len(golds)

    # BLEURT
    try:
        if 'bleurt_scorer' not in _MODEL_CACHE:
            print("Loading BLEURT-20 model...")
            from evaluate import load as hf_load
            _MODEL_CACHE['bleurt_scorer'] = hf_load("bleurt", "BLEURT-20", module_type="metric")
        bleurt_scores = _MODEL_CACHE['bleurt_scorer'].compute(predictions=predicteds, references=golds)['scores']
        results['BLEURT'] = bleurt_scores # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate BLEURT. Error: {e}")
        results['BLEURT'] = [-1.0] * len(golds)

    # COMET
    try:
        if 'comet_model' not in _MODEL_CACHE:
            print("Loading COMET-22 model...")
            import comet
            model_path = comet.download_model("Unbabel/wmt22-comet-da")
            _MODEL_CACHE['comet_model'] = comet.load_from_checkpoint(model_path)
        comet_data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(originals, predicteds, golds)]
        comet_output = _MODEL_CACHE['comet_model'].predict(comet_data, batch_size=2, gpus=1, progress_bar=False)
        results['COMET-22'] = comet_output.scores # <-- CHANGED: Store the list of scores
    except Exception as e:
        warnings.warn(f"Could not calculate COMET-22. Error: {e}")
        results['COMET-22'] = [-1.0] * len(golds)

    # --- 4. Format Output ---
    if not is_corpus:
        # For a single sentence, extract the single score from the list
        return {
            'BLEU': round(results['BLEU'], 4),
            'chrF++': round(results['chrF++'], 4),
            'TER (inv)': round(results['TER (inv)'], 4),
            'BERTScore': round(results['BERTScore'][0], 4),
            'BLEURT': round(results['BLEURT'][0], 4),
            'COMET-22': round(results['COMET-22'][0], 4),
        }
    else:
        # For a corpus, return corpus scores and lists of sentence scores
        return {
            'BLEU': {'corpus_score': round(results.get('BLEU', -1), 4)},
            'chrF++': {'corpus_score': round(results.get('chrF++', -1), 4)},
            'TER (inv)': {'corpus_score': round(results.get('TER (inv)', -1), 4)},
            'BERTScore': {'scores': [round(s, 4) for s in results.get('BERTScore', [])]},
            'BLEURT': {'scores': [round(s, 4) for s in results.get('BLEURT', [])]},
            'COMET-22': {'scores': [round(s, 4) for s in results.get('COMET-22', [])]},
        }

In [6]:
# read the dataset with the llm traslations


translated_dataset = pd.read_csv("/kaggle/input/popuss/predicted_opus.csv")

In [7]:
import os
import pandas as pd

file_paths = [
    "/kaggle/input/predicted/predicted_mistral.xlsx",
    "/kaggle/input/predicted/predicted_opus.csv"
]
modes = ["grammatic", "semantic"]
for mode in modes:
    for path in file_paths:
        if path.endswith(".xlsx"):
            translated_dataset = pd.read_excel(path)
            translated_dataset = translated_dataset.rename(columns={"Mistral Traslation":"prediction"})
        elif path.endswith(".csv"):
            translated_dataset = pd.read_csv(path)
        else:
            raise ValueError(f"Unsupported file type: {path}")
        translated_dataset = translated_dataset
        # Judging
        scores = []
        feedbacks = []
        print(f"\n\nJudging {path}, {mode}\n\n")
        for idx, row in translated_dataset.iterrows():
            to_predict = row["Sentence"]
            gold      = row["TRADUZIONE ITALIANA"]
            prediction = row["prediction"]
    
            judgement = judge(
                to_predict,
                prediction,
                gold,
                mode=mode,
                verbose=1
            )
            scores.append(judgement["score"])
            feedbacks.append(judgement["feedback"])
        
        # e.g. inspect or save
        mt_metrics = evaluate_metrics(
            translated_dataset["Sentence"].to_list(),
            translated_dataset["TRADUZIONE ITALIANA"].to_list(),
            translated_dataset["prediction"].to_list()
        )
        translated_dataset['Score']    = scores
        translated_dataset['Feedback'] = feedbacks
        translated_dataset['BERTScore'] = mt_metrics['BERTScore']['scores']
        translated_dataset['COMET-22']  = mt_metrics['COMET-22']['scores']
        translated_dataset['BLEURT']    = mt_metrics['BLEURT']['scores']
        translated_dataset['BLEU']      = pd.Series(mt_metrics['BLEU']['corpus_score'], index=translated_dataset.index[[0]])
        translated_dataset['chrF++']    = pd.Series(mt_metrics['chrF++']['corpus_score'], index=translated_dataset.index[[0]])
        translated_dataset['TER (inv)']= pd.Series(mt_metrics['TER (inv)']['corpus_score'], index=translated_dataset.index[[0]])
    
        # --- DYNAMIC OUTPUT PATH ---
        base = os.path.splitext(os.path.basename(path))[0]              # e.g. "predicted_mistral"
        output_path = f"/kaggle/working/{base}_traduzioni_con_score_{mode}.xlsx"  # e.g. "predicted_mistral_traduzioni_con_score.xlsx"
        translated_dataset.to_excel(output_path, index=False)
    
        print(f"\nResults for {path} (saved to {output_path}):")
        print(translated_dataset[["Sentence", "prediction", "Score"]].head())
        print("\n\n\n\n\n\n")




Judging /kaggle/input/predicted/predicted_mistral.xlsx, grammatic


ORIGINAL:
 quella guerra ben fatta l' opera perché etc. Et dall' altra parte Aiaces era uno cavaliere franco e prode all' arme, di gran guisa, ma non era pieno di grande senno
PREDICTION:
Quella guerra fu condotta con grande abilità, ecc. Dall'altra parte, Aiace era un cavaliere coraggioso e abile nella guerra, di grande aspetto, ma non era pieno di grande senno. 
GOLD:
 Quella guerra era ben condotta per l’impresa, eccetera. E d’altra parte Aiace era un cavaliere valoroso e audace in battaglia, di grande forza, ma non era dotato di grande saggezza
▶ Generating judgement ...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


SCORE: 4

ORIGINAL:
 crudele, e di tutte le colpe pigli vendetta, come dice la legge, ed a neuno cavaliere perdoni che pecchi.
PREDICTION:
crudele, e di tutte le colpe prendi vendetta, come dice la legge, e non perdonare a nessun cavaliere che pecchi. 
GOLD:
 Crudele, e si vendica di tutte le colpe, come prescrive la legge, e non perdona nessun cavaliere che sbagli
▶ Generating judgement ...
SCORE: 4

ORIGINAL:
 Non d' altra forza d' animo fue ornato Ponzio Aufidiano, romano cavaliere.
PREDICTION:
Non fu ornato di altro coraggio che Ponzio Aufidiano, cavaliere romano. 
GOLD:
 Ponzio Aufidiano, cavaliere romano, non fu dotato di minore forza d’animo.
▶ Generating judgement ...
SCORE: 4

ORIGINAL:
 Se questo piace a tutti e se 'l tempo hae bisogno d'avere Pompeio per cavaliere e non per compagno, non riterrò più i fati.
PREDICTION:
Se questo piace a tutti e se il tempo richiede che Pompeo sia il cavaliere e non il compagno, non mi opporrò più al fato. 
GOLD:
 Se questa è la volontà comun

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...


Results for /kaggle/input/predicted/predicted_mistral.xlsx (saved to /kaggle/working/predicted_mistral_traduzioni_con_score_semantic.xlsx):
                                            Sentence  \
0  quella guerra ben fatta l' opera perché etc. E...   
1  crudele, e di tutte le colpe pigli vendetta, c...   
2  Non d' altra forza d' animo fue ornato Ponzio ...   
3  Se questo piace a tutti e se 'l tempo hae biso...   
4  Officio di questa arte pare che sia dicere app...   

                                          prediction  Score  
0  Quella guerra fu condotta con grande abilità, ...      3  
1  crudele, e di tutte le colpe prendi vendetta, ...      2  
2  Non fu ornato di altro coraggio che Ponzio Auf...      2  
3  Se questo piace a tutti e se il tempo richiede...      3  
4  Sembra che il compito di questa arte sia quell...      3  









Judging /kaggle/input/predicted/predicted_opus.csv, semantic


ORIGINAL:
 quella guerra ben fatta l' opera perché etc. Et dall' altra parte A

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


SCORE: 3

ORIGINAL:
 crudele, e di tutte le colpe pigli vendetta, come dice la legge, ed a neuno cavaliere perdoni che pecchi.
PREDICTION:
crudele, e vendetta per tutti i peccati, come dice la legge, e non perdonare nessun cavaliere per il peccato. 
GOLD:
 Crudele, e si vendica di tutte le colpe, come prescrive la legge, e non perdona nessun cavaliere che sbagli
▶ Generating judgement ...
SCORE: 2

ORIGINAL:
 Non d' altra forza d' animo fue ornato Ponzio Aufidiano, romano cavaliere.
PREDICTION:
Ponzio Aufidiano, un cavaliere romano, era adornato senza altra forza della mente. 
GOLD:
 Ponzio Aufidiano, cavaliere romano, non fu dotato di minore forza d’animo.
▶ Generating judgement ...
SCORE: 3

ORIGINAL:
 Se questo piace a tutti e se 'l tempo hae bisogno d'avere Pompeio per cavaliere e non per compagno, non riterrò più i fati.
PREDICTION:
Se questo piace a tutti noi e se c'è bisogno di tempo per avere Pompeo come cavaliere e non come compagno, non considererò più i destini. 
GOLD:
 Se q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Results for /kaggle/input/predicted/predicted_opus.csv (saved to /kaggle/working/predicted_opus_traduzioni_con_score_semantic.xlsx):
                                            Sentence  \
0  quella guerra ben fatta l' opera perché etc. E...   
1  crudele, e di tutte le colpe pigli vendetta, c...   
2  Non d' altra forza d' animo fue ornato Ponzio ...   
3  Se questo piace a tutti e se 'l tempo hae biso...   
4  Officio di questa arte pare che sia dicere app...   

                                          prediction  Score  
0  che la guerra ha fatto bene perché, ecc. Et, d...      3  
1  crudele, e vendetta per tutti i peccati, come ...      2  
2  Ponzio Aufidiano, un cavaliere romano, era ado...      3  
3  Se questo piace a tutti noi e se c'è bisogno d...      3  
4  L'ufficio di questa arte sembra essere quello ...      3  









# Kappa

In [8]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np


In [9]:
idk = {"grammar":"grammatic", "semantic":"semantic"}
for model in ("mistral", "opus"):
    for mode in ("grammar", "semantic"):
        human_df = pd.read_csv(f"/kaggle/input/human-scores-predictionsdd/predicted_{model}_human_score.csv")
        ai_df = pd.read_excel(f"/kaggle/working/predicted_{model}_traduzioni_con_score_{idk[mode]}.xlsx")
        ai_df["human_score"] = human_df["human_"+mode]
        kappa = cohen_kappa_score(human_df["human_"+mode], ai_df["Score"])
        kappa_linear = cohen_kappa_score(human_df["human_"+mode], ai_df["Score"], weights="linear")
        kappa_quadratic = cohen_kappa_score(human_df["human_"+mode], ai_df["Score"], weights="quadratic")
        ai_df['kappa'] = pd.Series(kappa, index=ai_df.index[[0]])
        ai_df['kappa_linear'] = pd.Series(kappa_linear, index=ai_df.index[[0]])
        ai_df['kappa_quadratic'] = pd.Series(kappa_quadratic, index=ai_df.index[[0]])
        ai_df.to_csv(f"/kaggle/working/predicted_{model}_traduzioni_con_score_{idk[mode]}.csv")


In [10]:
ai_df

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,Unnamed: 0,Author,Date,TESTO,Region,MOVIMENTO,Sentence,TRADUZIONE ITALIANA,prediction,Score,...,BERTScore,COMET-22,BLEURT,BLEU,chrF++,TER (inv),human_score,kappa,kappa_linear,kappa_quadratic
0,0,Brunetto Latini,1260-61,Tesoro,fior.,Preumanesimo,quella guerra ben fatta l' opera perché etc. E...,"Quella guerra era ben condotta per l’impresa, ...","che la guerra ha fatto bene perché, ecc. Et, d...",3,...,0.8466,0.6510,0.3861,0.2233,0.4858,0.3625,3,0.098206,0.216478,0.320784
1,1,Bono Giamboni,1292,Il libro de' vizi e delle virtudi,fior.,Preumanesimo,"crudele, e di tutte le colpe pigli vendetta, c...","Crudele, e si vendica di tutte le colpe, come ...","crudele, e vendetta per tutti i peccati, come ...",2,...,0.8049,0.7437,0.4946,NaN,NaN,NaN,5,NaN,NaN,NaN
2,2,Valerio Massimo,1336,Nove libri di fatti e detti memorabili,fior.,Letteratura Enciclopedica Didattica Medievale,Non d' altra forza d' animo fue ornato Ponzio ...,"Ponzio Aufidiano, cavaliere romano, non fu dot...","Ponzio Aufidiano, un cavaliere romano, era ado...",3,...,0.8663,0.6349,0.4154,NaN,NaN,NaN,3,NaN,NaN,NaN
3,3,Luciano,1330/40,Dialoghi (traduzione dal Greco),prat.,Traduzione volgare testo greco,Se questo piace a tutti e se 'l tempo hae biso...,"Se questa è la volontà comune, e le circostanz...",Se questo piace a tutti noi e se c'è bisogno d...,3,...,0.7544,0.6611,0.4573,NaN,NaN,NaN,4,NaN,NaN,NaN
4,4,Brunetto Latini,1260-61,Tesoro,fior.,Preumanesimo,Officio di questa arte pare che sia dicere app...,Il compito di quest’arte sembra essere il parl...,L'ufficio di questa arte sembra essere quello ...,3,...,0.8185,0.6104,0.5894,NaN,NaN,NaN,4,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,92,Grazia di Meo,1343,NaN,tosc.,letteratura scientifico-morale medievale,"Alchuno è riccho e gentile, ma lamentasi che v...","Qualcuno è ricco e gentile, ma si lamenta perc...","Qualcuno è ricco e gentile, ma si è lamentato ...",3,...,0.8751,0.8742,0.6386,NaN,NaN,NaN,2,NaN,NaN,NaN
93,93,Zanobi da Strada,1361,NaN,tosc.,Letteratura religiosa medievale,"Pietro, essendogli mostrato in figura il popul...","A Pietro, cui fu mostrato in visione il popolo...","Pietro, mostrandogli nella sua figura il popol...",2,...,0.8430,0.6953,0.5983,NaN,NaN,NaN,4,NaN,NaN,NaN
94,94,Declamazioni di Seneca,1392,NaN,tosc.,NaN,pregollo che lo liberasse di quella obbligazio...,Gli chiese di liberarlo da quell’obbligazione ...,"L'ho pregato di liberarlo da quell'obbligo, ne...",2,...,0.8485,0.7417,0.4811,NaN,NaN,NaN,1,NaN,NaN,NaN
95,95,Giovanni da San Miniato,1415,NaN,tosc.,Letteratura religiosa medievale,L'oro verrà dall'Aquilone. Che figuriamo noi p...,L’oro verrà dall’Aquilone . Che cosa rappresen...,L'oro verrà dall'aquilone. Consideriamolo per ...,3,...,0.8469,0.7383,0.6377,NaN,NaN,NaN,2,NaN,NaN,NaN


In [11]:
t = pd.read_csv("/kaggle/working/predicted_mistral_traduzioni_con_score_grammatic.csv")
t

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,Unnamed: 0.1,Unnamed: 0,Author,Date,TESTO,Region,MOVIMENTO,Sentence,TRADUZIONE ITALIANA,prediction,...,BERTScore,COMET-22,BLEURT,BLEU,chrF++,TER (inv),human_score,kappa,kappa_linear,kappa_quadratic
0,0,0,Brunetto Latini,1260-61,Tesoro,fior.,Preumanesimo,quella guerra ben fatta l' opera perché etc. E...,"Quella guerra era ben condotta per l’impresa, ...","Quella guerra fu condotta con grande abilità, ...",...,0.8538,0.7937,0.6002,0.3326,0.5762,0.4737,5,0.019602,0.053996,0.115117
1,1,1,Bono Giamboni,1292,Il libro de' vizi e delle virtudi,fior.,Preumanesimo,"crudele, e di tutte le colpe pigli vendetta, c...","Crudele, e si vendica di tutte le colpe, come ...","crudele, e di tutte le colpe prendi vendetta, ...",...,0.8388,0.7810,0.6234,NaN,NaN,NaN,4,NaN,NaN,NaN
2,2,2,Valerio Massimo,1336,Nove libri di fatti e detti memorabili,fior.,Letteratura Enciclopedica Didattica Medievale,Non d' altra forza d' animo fue ornato Ponzio ...,"Ponzio Aufidiano, cavaliere romano, non fu dot...",Non fu ornato di altro coraggio che Ponzio Auf...,...,0.8476,0.7335,0.4373,NaN,NaN,NaN,3,NaN,NaN,NaN
3,3,3,Luciano,1330/40,Dialoghi (traduzione dal Greco),prat.,Traduzione volgare testo greco,Se questo piace a tutti e se 'l tempo hae biso...,"Se questa è la volontà comune, e le circostanz...",Se questo piace a tutti e se il tempo richiede...,...,0.7887,0.7604,0.5674,NaN,NaN,NaN,5,NaN,NaN,NaN
4,4,4,Brunetto Latini,1260-61,Tesoro,fior.,Preumanesimo,Officio di questa arte pare che sia dicere app...,Il compito di quest’arte sembra essere il parl...,Sembra che il compito di questa arte sia quell...,...,0.8203,0.8577,0.8317,NaN,NaN,NaN,5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,92,92,Grazia di Meo,1343,NaN,tosc.,letteratura scientifico-morale medievale,"Alchuno è riccho e gentile, ma lamentasi che v...","Qualcuno è ricco e gentile, ma si lamenta perc...",Non c'è nessuno ricco e gentile che non si lam...,...,0.8389,0.8589,0.7163,NaN,NaN,NaN,4,NaN,NaN,NaN
93,93,93,Zanobi da Strada,1361,NaN,tosc.,Letteratura religiosa medievale,"Pietro, essendogli mostrato in figura il popul...","A Pietro, cui fu mostrato in visione il popolo...","Pietro, una volta mostratogli il popolo pagano...",...,0.8775,0.8289,0.6474,NaN,NaN,NaN,5,NaN,NaN,NaN
94,94,94,Declamazioni di Seneca,1392,NaN,tosc.,NaN,pregollo che lo liberasse di quella obbligazio...,Gli chiese di liberarlo da quell’obbligazione ...,Egli mi pregò di liberarlo da quell'obbligo in...,...,0.8401,0.7996,0.6741,NaN,NaN,NaN,4,NaN,NaN,NaN
95,95,95,Giovanni da San Miniato,1415,NaN,tosc.,Letteratura religiosa medievale,L'oro verrà dall'Aquilone. Che figuriamo noi p...,L’oro verrà dall’Aquilone . Che cosa rappresen...,L'oro proviene da Aquilone. Che noi figuriamo ...,...,0.8878,0.7626,0.5369,NaN,NaN,NaN,4,NaN,NaN,NaN
